In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

In [3]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [4]:
BASE_URL = f"https://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

In [5]:
def fetch(url):
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            return response.text
        return None
    except:
        return None

In [6]:
def parse_content(html, url):
    soup = BeautifulSoup(html, "lxml")

    for book in soup.select("article.product_pod"):
        title = book.h3.a["title"]
        price = book.select_one(".price_color").get_text(strip=True)
        price = float(price.replace("Â£", ""))
        availability = book.select_one(".instock.availability").get_text(strip=True)
        image_src = requests.compat.urljoin(url, book.select_one("img")["src"])
        content.append(
            {
                "title": title,
                "price": price,
                "availability": availability,
                "image_src": image_src,
            }
        )

In [7]:
def download_image(image_url, destination_folder):
    os.makedirs(destination_folder, exist_ok=True)
    file_name = os.path.join(destination_folder, image_url.split("/")[-1])

    response = requests.get(image_url, headers=HEADERS, stream=True)
    with open(file_name, "wb") as fh:
        for chunk in response.iter_content(1024):
            fh.write(chunk)

In [8]:
content = []
page_number = 1
for i in range(2):
    full_url = BASE_URL + f"catalogue/page-{page_number}.html"
    html_string = fetch(full_url)
    if html_string:
        parse_content(html_string, full_url)
        page_number += 1
        time.sleep(1)
    else:
        continue

In [9]:
df = pd.DataFrame(content)
print(df.shape)
df.head()

(40, 4)


,title,price,availability,image_src
0,A Light in the Attic,51.77,In stock,https://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg
1,Tipping the Velvet,53.74,In stock,https://books.toscrape.com/media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg
2,Soumission,50.10,In stock,https://books.toscrape.com/media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg
3,Sharp Objects,47.82,In stock,https://books.toscrape.com/media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg
4,Sapiens: A Brief History of Humankind,54.23,In stock,https://books.toscrape.com/media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg


In [ ]:
cheap_books_df = df[df["price"] < 15]
cheap_books_df.shape

for book in cheap_books_df.itertuples():
    if book.image_src:
        download_image(book.image_src, "books_images")
        print(f"Downloaded image for {book.title}")

Pandas(Index=10, title='Starving Hearts (Triangular Trade Trilogy, #1)', price=13.99, availability='In stock', image_src='https://books.toscrape.com/media/cache/be/f4/bef44da28c98f905a3ebec0b87be8530.jpg')
Pandas(Index=20, title='In Her Wake', price=12.84, availability='In stock', image_src='https://books.toscrape.com/media/cache/5d/72/5d72709c6a7a9584a4d1cf07648bfce1.jpg')
